[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_GPU/CUDA_Cpp.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# CUDA in C++

> ⚠️ **Draft — code not machine-verified.** Requires an NVIDIA GPU + the CUDA toolkit (`nvcc`) not available at authoring time. **Colab's free GPU runtime does include `nvcc`** (Runtime → Change runtime type → T4 GPU), but the blocks below are fenced C++ meant to be compiled locally (`nvcc file.cu -o prog && ./prog`), not executable notebook cells — on Colab, paste each one into a `%%writefile file.cu` cell followed by a `!nvcc file.cu -o prog && ./prog` cell. An instructor should run each block (locally or in Colab) before teaching. Remove this banner after that pass.

For students who outgrow Numba: raw CUDA C++ — explicit memory management, kernel launches, error handling, and the library ecosystem (cuBLAS/cuFFT) that usually beats hand-written kernels. Compile everything with `nvcc file.cu -o prog`.

## 1. Pre-requisites

[Intro to C](../Intro_Programming/Intro_C.ipynb) (pointers, malloc discipline); [Hardware-Accelerated Computing](./HW_Accelerated_Computing.ipynb) for the concepts.

---
### 🕐 Session 1 of 3 — *First Kernels* (~40 min)
**Goal:** the full lifecycle: allocate, copy, launch, synchronize, check errors, free.
**Feeds into:** Session 2 (memory management).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: First Kernels</b></summary>

**Timing (~40 min).** 5 min the draft caveat and toolchain · 10 min two heaps · 12 min the lifecycle · 13 min error checking.

**Practical warning first: nothing here is executed, and it needs `nvcc`.** The banner says so. **Compile and run every block yourself before teaching** — `nvcc -O3 saxpy.cu -o saxpy && ./saxpy` — because toolkit version mismatches and missing `-arch` flags fail in ways that are miserable to debug in front of a room.

**Frame the session by what students are trading, since they arrive from Numba.** Numba gave them kernels for free: allocation, transfer, and launch all implicit. CUDA C++ makes **every one of those explicit**. **You are not learning new concepts — you are learning to write down concepts you already have**, and the payoff is being able to read any CUDA codebase.

**Make the two-heap point the session's first real idea, because it is where the bugs live.** Host and device pointers are both `float*`. **The type system cannot tell them apart.** Dereference a device pointer on the host and you segfault; pass a host pointer to a kernel and you get silent garbage. **Both compile cleanly**, which is exactly what makes it dangerous.

**Give the room the convention professionals use as a substitute for type safety.** Suffix everything — `x_h` for host, `x_d` for device — which is precisely what this code does. **The compiler will not help you, so the variable name has to.** It feels fussy for five minutes and prevents a class of bug that costs hours.

**Then walk the six-step lifecycle and have the room recite it back.** `cudaMalloc` → `cudaMemcpy` (H2D) → launch → `cudaDeviceSynchronize` → `cudaMemcpy` (D2H) → `cudaFree`. **Numba performed all six invisibly.** Step 4 is not optional: launches are asynchronous, so without it you may copy back a buffer the kernel has not finished writing.

**Make `CUDA_CHECK` the centrepiece, because the notebook is right that it is not style.** **CUDA errors are returned, not thrown.** Ignore the return value and a failed launch produces no message — execution continues and the results are silently wrong. **A CUDA program without error checking does not report failure; it reports garbage**, and that is the sharpest difference from ordinary C++.

**Point out that this code checks in *two* places, and say why, because students usually do one.** A launch `saxpy<<<g,b>>>(...)` **returns void** and cannot be wrapped. So the idiom is `cudaGetLastError()` immediately after (catches bad launch configurations — too many threads, too much shared memory) **and** `cudaDeviceSynchronize()` (catches faults during execution). **Two failure windows, two checks**, and the file demonstrates both.

**Point at the index arithmetic and connect it back explicitly.** `blockIdx.x * blockDim.x + threadIdx.x` is character-for-character what `cuda.grid(1)` computed in [Hardware-Accelerated Computing](./HW_Accelerated_Computing.ipynb), and `(N + 255) / 256` is ceiling division — the same role as `triton.cdiv`. **The `if (i < n)` guard exists because that ceiling over-provisions the last block.** On a GPU an out-of-bounds write does not fault; it corrupts memory silently.

**Close by placing this workshop honestly.** Most people should write Numba, Triton, or library calls. **CUDA C++ is for reading other people's code, for the last 20% of performance, and for operations nothing else can express.** That is a real but narrow niche, and saying so is more useful than presenting it as the destination.
</details>

💡 **Intuition.** CUDA C++ is [Intro to C's](../Intro_Programming/Intro_C.ipynb) memory discipline with *two* heaps: host pointers and device pointers are the same type (`float*`) but live in different worlds, and mixing them up compiles fine then crashes at runtime. The `CUDA_CHECK` macro habit below is not optional style — kernel launches fail *silently* without it.

```cpp
// saxpy.cu — the "hello world" of CUDA
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e)); \
        exit(1); } } while (0)

__global__ void saxpy(int n, float a, const float* x, float* y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // same arithmetic as Numba's cuda.grid(1)
    if (i < n) y[i] = a * x[i] + y[i];
}

int main() {
    const int N = 1 << 24;
    float *x_h = (float*)malloc(N * sizeof(float));
    float *y_h = (float*)malloc(N * sizeof(float));
    for (int i = 0; i < N; ++i) { x_h[i] = 1.0f; y_h[i] = 2.0f; }

    float *x_d, *y_d;
    CUDA_CHECK(cudaMalloc(&x_d, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&y_d, N * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(x_d, x_h, N * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(y_d, y_h, N * sizeof(float), cudaMemcpyHostToDevice));

    saxpy<<<(N + 255) / 256, 256>>>(N, 3.0f, x_d, y_d);
    CUDA_CHECK(cudaGetLastError());                  // catches bad launch configs
    CUDA_CHECK(cudaDeviceSynchronize());             // kernels are ASYNC: wait before trusting results

    CUDA_CHECK(cudaMemcpy(y_h, y_d, N * sizeof(float), cudaMemcpyDeviceToHost));
    printf("y[0] = %f (expect 5.0)\n", y_h[0]);
    cudaFree(x_d); cudaFree(y_d); free(x_h); free(y_h);
}
```

Build & run: `nvcc -O3 saxpy.cu -o saxpy && ./saxpy`

---
### 🕐 Session 2 of 3 — *Memory Management Patterns* (~35 min)
**Goal:** unified vs explicit memory, pinned transfers, and measuring bandwidth honestly.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (libraries).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Memory Management Patterns</b></summary>

**Timing (~35 min).** 12 min the three styles · 10 min why unified memory is unpredictable · 13 min the benchmark ritual.

**Present the three styles as a ladder of control, and say that the right rung depends on the project's stage.** **Unified** (`cudaMallocManaged`) is one pointer valid on both sides — simplest code, least predictable performance. **Explicit** device memory plus ordinary host memory is the default. **Pinned** host memory (`cudaMallocHost`) is page-locked, roughly 2× faster to transfer, and the prerequisite for asynchronous overlap.

**Explain what unified memory actually does, because "it just works" hides the cost.** The driver migrates **pages on demand**: touch data on the host, the page faults across; touch it on the device, it faults back. **A loop alternating host and device access can thrash pages back and forth**, and the resulting slowdown appears nowhere in your source. **Great for prototyping, profile before shipping** — the notebook's advice is exactly right and worth repeating.

**Then pinned memory, and give the reason rather than the recipe.** DMA hardware needs physical addresses that will not move. Ordinary host allocations are **pageable** — the [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) may relocate or swap them at any moment — so CUDA first copies into a hidden pinned staging buffer. **That staging copy is synchronous, which is why pageable memory cannot overlap.** Pinning removes it and makes `cudaMemcpyAsync` genuinely asynchronous, which is what the [streams session](./HW_Accelerated_Computing.ipynb) depends on.

**Name the cost immediately, so nobody pins everything.** Pinned pages **cannot be swapped out**, so they consume real RAM and pressure the OS; over-pinning degrades the whole machine. **Pin the buffers you stream through, not your dataset.** This is exactly what `pin_memory=True` requests in a PyTorch `DataLoader` — where most students have already met it without knowing why.

**Make the benchmark ritual the session's methodological content, because it is a skill that transfers.** Note that this code gets it **right**: `cudaEventRecord` around the transfer and `cudaEventSynchronize` before reading the elapsed time. **CUDA events time device work on the device**, which wall-clock cannot do reliably because launches are asynchronous.

**Contrast it directly with the counterexample elsewhere in this topic, since having both is a teaching asset.** The CuPy timings in [Intro to GPU Systems §3.3](./Intro_GPU.ipynb) use `time.perf_counter()` with no synchronisation and therefore measure launch overhead — which is how that notebook reaches an apparent 75,000× speedup. **Same topic, two patterns, one of them right.**

**Add the two rules the snippet does not show, because a single timed transfer is not a measurement.** **Warm up** — the first CUDA call in a process pays context initialisation, measured at 0.147 s in [Intro to GPU Systems](./Intro_GPU.ipynb). And **repeat, taking the minimum**, as [Performance Engineering](./Performance_Engineering.ipynb)'s `bench` helper does, since every source of interference makes a run slower and never faster.

**Give them a target number, because a measurement with no expected value is not a check.** The comment says ~6 GB/s pageable and ~12 GB/s pinned; PCIe 4.0 ×16 tops out near 32 GB/s theoretical. **If you record 3 GB/s, something is wrong** — wrong slot, wrong generation, or a hidden staging copy — and knowing the target is what tells you to investigate.

**Close by connecting to the decision this all serves.** [Intro to GPU Systems](./Intro_GPU.ipynb) established that a round trip for 80 MB costs ~30 ms, so any computation finishing faster is not worth sending. **Halving the transfer cost with pinned memory halves that threshold.** Memory management does not accelerate arithmetic; it lowers the bar for what is worth accelerating at all.
</details>

**The three memory styles**, in ascending control:

```cpp
// 1. Unified memory — one pointer works on both sides; the driver migrates pages on demand
float* u; cudaMallocManaged(&u, N * sizeof(float));
// simplest code, unpredictable performance: great for prototyping, profile before shipping

// 2. Explicit device memory + regular host memory (Session 1) — the default

// 3. Pinned (page-locked) host memory — DMA-able, ~2x faster transfers, enables async overlap
float* p; cudaMallocHost(&p, N * sizeof(float));
cudaMemcpyAsync(x_d, p, N * sizeof(float), cudaMemcpyHostToDevice, stream);
```

Benchmark ritual (do this on your machine and keep the numbers):

```cpp
cudaEvent_t t0, t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
cudaEventRecord(t0);
cudaMemcpy(x_d, x_h, N * sizeof(float), cudaMemcpyHostToDevice);
cudaEventRecord(t1); cudaEventSynchronize(t1);
float ms; cudaEventElapsedTime(&ms, t0, t1);
printf("H2D: %.1f GB/s\n", N * sizeof(float) / ms / 1e6);
// pageable vs pinned typically ~6 vs ~12 GB/s on PCIe 4 — measure yours
```

---
### 🕐 Session 3 of 3 — *The Library Ecosystem* (~35 min)
**Goal:** stop writing kernels: cuBLAS matmul and cuFFT spectra, correctly linked and checked.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Library Ecosystem</b></summary>

**Timing (~35 min).** 10 min the production rule · 10 min the column-major gotcha · 8 min cuFFT · 7 min the homework.

**Open with the rule and then justify it, because it appears to undercut the previous two sessions.** *Kernel-write only what cuBLAS, cuFFT, cuDNN, and Thrust do not already do.* Students who have just hand-tiled a matmul reasonably ask what that was for. **The answer is that your tiled kernel exists so you understand theirs** — you now know what register blocking, tiling, and occupancy mean, which is what lets you read a profiler and decide whether a library call is performing.

**Make the size of the gap concrete, since that is what makes the rule stick.** NVIDIA's GEMM has had two decades of work: register blocking, vectorised loads, architecture-specific tile shapes, tensor cores, and hand-written assembly for the inner loop. **A good hand-written tiled kernel typically reaches 20–40% of cuBLAS.** That is not a failure of the student; it is the correct expectation, and it should be stated before the homework rather than discovered as a disappointment.

**Then the column-major gotcha, which is the single most common cuBLAS bug and deserves its own slot.** cuBLAS inherits Fortran conventions: **matrices are column-major**, while C, NumPy, and PyTorch are row-major. The trick in the code — computing $B^TA^T$ with swapped arguments — yields $A \cdot B$ in row-major **with no transposes and no copies**, because a row-major matrix *is* its own transpose read column-major.

**Have the room verify that identity rather than accepting it, because it is genuinely confusing.** $(AB)^T = B^TA^T$; a row-major buffer interpreted column-major is the transpose; so asking cuBLAS for $B^TA^T$ in its own convention and reading the result row-major gives $AB$. **Note the argument order in the call: `B_d` comes before `A_d`, and `n`, `m`, `k` are permuted.** If a cuBLAS result is transposed or garbage, this is where to look first.

**Give cuFFT one honest sentence about what a "plan" is.** `cufftPlan1d` precomputes twiddle factors and chooses an algorithm for that specific size — so **creating a plan is expensive and executing it is cheap**. Create once, execute many times. **Creating a plan inside a loop is the classic cuFFT performance bug**, and it is the same idea as FFTW's wisdom.

**Point at `CUFFT_R2C` and connect it back to the DSP track.** A real-valued input has a conjugate-symmetric spectrum, so cuFFT returns only $N/2 + 1$ bins — **half the memory and half the work**, for free. That is the Hermitian symmetry from [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) showing up as an API choice, and it is worth naming as one.

**Flag the error-checking gap in these snippets, since Session 1 was emphatic about it.** `cublasCreate`, `cublasSgemm`, and the cuFFT calls all return status codes, and **none of them is checked here**. They need their own macros — `cublasStatus_t` and `cufftResult` are distinct types from `cudaError_t`. **The snippets are illustrative rather than production-ready**, and a room that just learned `CUDA_CHECK` should notice its absence.

**Set the homework properly, because the shape of the answer is the lesson.** Benchmark saxpy, the tiled matmul, and cuBLAS on the same sizes; plot GFLOP/s. **Expect saxpy to be memory-bound at a tiny fraction of peak, the tiled kernel to reach a modest share, and cuBLAS to approach the roof** — exactly the placement [Performance Engineering](./Performance_Engineering.ipynb) predicts from arithmetic intensity. **Two of the three gaps are explained by the roofline; the third is engineering effort**, and distinguishing them is the point.
</details>

💡 **Intuition.** NVIDIA's library engineers have spent two decades on tiled matmuls; your [tiled kernel](./HW_Accelerated_Computing.ipynb) exists so you *understand* theirs. Production rule: kernel-write only what cuBLAS/cuFFT/cuDNN/Thrust don't already do.

```cpp
// gemm.cu — C = A·B via cuBLAS (column-major! the eternal gotcha)
#include <cublas_v2.h>
cublasHandle_t h; cublasCreate(&h);
const float one = 1.0f, zero = 0.0f;
// note the trick: computing B^T·A^T in column-major = A·B in row-major
cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, n, m, k, &one, B_d, n, A_d, k, &zero, C_d, n);
cublasDestroy(h);
// link: nvcc gemm.cu -lcublas
```

```cpp
// spectrum.cu — cuFFT: the FFT from [Foundations 1 S7], at GB/s
#include <cufft.h>
cufftHandle plan;
cufftPlan1d(&plan, N, CUFFT_R2C, 1 /*batch*/);
cufftExecR2C(plan, signal_d, spectrum_d);        // in-place batched variants exist
cufftDestroy(plan);
// link: nvcc spectrum.cu -lcufft
```

Homework with teeth: benchmark your Session-1 saxpy, your tiled matmul, and cuBLAS on the
same sizes; plot GFLOP/s. The gap between your kernel and cuBLAS is the syllabus of a
graduate course — and the reason the library exists.

## 4. Conclusion

Explicit two-heap memory discipline, error-checked async launches, pinned transfers, and libraries first. You can now read any CUDA codebase — and know which parts you shouldn't write yourself.

---
## Where next

- [Hardware-Accelerated Computing](./HW_Accelerated_Computing.ipynb) — the performance levers, in Python where iteration is fast.
- [Intro to C](../Intro_Programming/Intro_C.ipynb) — the pointer discipline this workshop doubles down on.